## Imports

In [ ]:
import sys
import math
import yaml
import shutil
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

current_dir = Path.cwd()
src_dir = current_dir.parent / "src"
if str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))

try:
    from repo_paths import get_repo_paths
    from model_builder import build_model_from_config
    from callbacks import get_callbacks
except ImportError as e:
    print(f"Import Error: {e}")


## Setup and Loading of the Model Configuration

In [ ]:
# Get the relevant paths of this repo
repo_paths = get_repo_paths()

# Load the model config
config_path = repo_paths["config_dir"] / "model_config.yaml"
if not config_path.exists():
    raise FileNotFoundError(f"Config file not found at: {config_path}")
# Read the config file into a dictionary
with open(config_path, "r") as f:
    config = yaml.safe_load(f)


print(f"✅ Project Root: {repo_paths["repo_root"]}")
print(f"✅ TensorFlow Version: {tf.__version__}")
print(f"✅ Devices: {tf.config.list_physical_devices()}")

## Set up the directory where this training run's data will be stored

In [ ]:
def setup_experiment():
    """
    Creates a unique run directory organized by model type.
    Structure: runs / <model_family> / <run_name>_<timestamp>
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_name = config['project']['run_name']
    
    # 1. Determine Model Family Folder
    if config['model'].get('type') == 'transfer':
        # e.g., "mobilenetv3small"
        model_family = config['model']['transfer']['base_model'].lower()
    else:
        model_family = "custom"
        
    # 2. Construct Paths
    base_run_dir = repo_paths["runs_dir"] / model_family
    this_run_dir = base_run_dir / f"{run_name}_{timestamp}"
    
    # Create directories
    this_run_dir.mkdir(parents=True, exist_ok=True)
    
    log_dir = this_run_dir / "logs"
    log_dir.mkdir(exist_ok=True)
    
    checkpoint_dir = this_run_dir / "checkpoints"
    checkpoint_dir.mkdir(exist_ok=True)
    
    # Backup config
    shutil.copy(config_path, this_run_dir / "config.yaml")
        
    print(f"Experiment initialized at: {this_run_dir}")
    print(f"Model Family: {model_family}")
    
    return this_run_dir, log_dir, checkpoint_dir

# Run setup
run_dir, log_dir, checkpoint_dir = setup_experiment()

## Data Loading

In [ ]:
#  1. Configuration
BATCH_SIZE = config['data']['batch_size']
IMG_HEIGHT = config['data']['img_height']
IMG_WIDTH = config['data']['img_width']
CHANNELS = config['data']['channels']
SPLIT_DIR = repo_paths["patches_dir"]

COLOR_MODE = "grayscale" if CHANNELS == 1 else "rgb"
print(f"Loading Data from: {SPLIT_DIR}")
print(f"  Target Size: {IMG_HEIGHT}x{IMG_WIDTH} | Mode: {COLOR_MODE} | Batch: {BATCH_SIZE}")

# 2. Create Datasets
def load_dataset(subset_name, shuffle=True):
    return tf.keras.utils.image_dataset_from_directory(
        SPLIT_DIR / subset_name,
        labels="inferred",
        label_mode="binary",
        class_names=[repo_paths["class_name_neg"], repo_paths["class_name_pos"]], 
        color_mode=COLOR_MODE,
        batch_size=BATCH_SIZE,
        image_size=(IMG_HEIGHT, IMG_WIDTH),
        shuffle=shuffle, 
        seed=config['project']['seed'] if shuffle else None,
        verbose=True
    )

train_ds = load_dataset("train", shuffle=True)
val_ds = load_dataset("val", shuffle=False)
test_ds = load_dataset("test", shuffle=False)

print(f"\n Class Mapping: {train_ds.class_names}") 

# 3. Performance Optimization for limited RAM and VRAM (no .cache used)
train_ds = train_ds.prefetch(buffer_size=1)
val_ds = val_ds.prefetch(buffer_size=1)
test_ds = test_ds.prefetch(buffer_size=1)

print("\n Data Pipeline Optimized (Safe Mode)")

## Build & Compile the Model

In [ ]:
print(f"Building Model Architecture: {config['project']['name']}")

try:
    # Build the model 
    model = build_model_from_config(config)

    # Verify that the model's input shape matches the data
    expected_shape = (None, IMG_HEIGHT, IMG_WIDTH, CHANNELS)
    
    if model.input_shape != expected_shape:
        print(f"Warning: Model input shape {model.input_shape} does not match expected data shape {expected_shape}")
    else:
        print(f"Input Shape Verified: {model.input_shape}")

    # Display the architecture
    model.summary()

except Exception as e:
    print(f"Model Build Failed: {e}")
    raise e

## Callbacks and Training

In [ ]:
# --- PREPARE CALLBACKS ---
callbacks = get_callbacks(config, run_dir, log_dir, checkpoint_dir)

print(f"Starting Training for {config['train']['epochs']} Epochs...")

# --- START TRAINING ---
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=config['train']['epochs'],
    callbacks=callbacks,
    verbose=1
)

# --- SAVE FINAL STATE ---
# We save the final model state regardless of whether it was the "best"
final_model_path = run_dir / "final_model.keras"
model.save(final_model_path)
print(f"Training Complete! Final model saved to: {final_model_path.name}")

## Visualization

In [ ]:
def plot_history(history, save_dir=None):
    """
    Dynamically plots all metrics present in the history object 
    and saves the resulting figure to the run directory.
    """
    # 1. Identify unique base metrics (ignore 'val_' prefix)
    all_keys = history.history.keys()
    # Filter out 'val_' keys and 'lr' (learning rate) to focus on performance metrics
    base_metrics = [k for k in all_keys if not k.startswith('val_') and k != 'lr']
    
    if not base_metrics:
        print("No metrics found to plot.")
        return

    # 2. Determine Grid Size
    n_metrics = len(base_metrics)
    n_cols = 2
    n_rows = math.ceil(n_metrics / n_cols)
    
    # 3. Create Figure
    plt.figure(figsize=(12, 4 * n_rows))
    epochs_range = range(1, len(history.history[base_metrics[0]]) + 1)

    for i, metric in enumerate(base_metrics):
        plt.subplot(n_rows, n_cols, i + 1)
        
        # Plot Training Metric
        plt.plot(epochs_range, history.history[metric], label=f'Training {metric.title()}')
        
        # Plot Validation Metric (if exists)
        val_key = f"val_{metric}"
        if val_key in all_keys:
            plt.plot(epochs_range, history.history[val_key], label=f'Validation {metric.title()}', linestyle='--')
            
        plt.legend(loc='best')
        plt.title(f'{metric.title()}')
        plt.xlabel('Epochs')
        plt.grid(True, alpha=0.3)
        
        # Intelligent ticks
        if len(epochs_range) <= 20:
            plt.xticks(epochs_range[::2])
    
    # Plot Learning Rate separately if it exists (Optional but helpful)
    if 'lr' in all_keys:
        plt.figure(figsize=(10, 3))
        plt.plot(epochs_range, history.history['lr'], color='orange', label='Learning Rate')
        plt.title("Learning Rate Schedule")
        plt.xlabel("Epochs")
        plt.ylabel("LR")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()

    plt.tight_layout()
    
    # 4. Save to Run Directory
    if save_dir:
        save_path = save_dir / "training_metrics.png"
        plt.savefig(save_path)
        print(f"Plots saved to: {save_path}")
        
    plt.show()

# Run the plotting function
plot_history(history, save_dir=run_dir)

### Visualization from .csv in case history is no longer available

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import math
from pathlib import Path

def plot_history_from_csv(csv_log_path, save_dir=None):
    """
    Dynamically plots all metrics present in the training log CSV
    and saves the resulting figure to the run directory.
    """
    # 0. Load the CSV log
    try:
        df = pd.read_csv(csv_log_path)
    except FileNotFoundError:
        print(f"⚠️ Log file not found at: {csv_log_path}")
        return

    # Create a dictionary similar to Keras history.history
    # We exclude 'epoch' from the metrics dict
    history = {}
    for col in df.columns:
        if col != 'epoch':
            history[col] = df[col].tolist()

    # 1. Identify unique base metrics (ignore 'val_' prefix)
    all_keys = history.keys()
    # Filter out 'val_' keys and 'lr' to focus on performance metrics
    base_metrics = [k for k in all_keys if not k.startswith('val_') and k != 'lr']
    
    if not base_metrics:
        print("⚠️ No metrics found to plot.")
        return

    # 2. Determine Grid Size
    n_metrics = len(base_metrics)
    n_cols = 2
    n_rows = math.ceil(n_metrics / n_cols)
    
    # 3. Create Main Figure for Metrics
    main_fig = plt.figure(figsize=(12, 4 * n_rows))
    
    # Use 'epoch' column for x-axis if available, otherwise 1-based index
    if 'epoch' in df.columns:
        # Keras CSVLogger is 0-indexed, usually we plot 1-indexed
        epochs_range = df['epoch'] + 1
    else:
        epochs_range = range(1, len(history[base_metrics[0]]) + 1)

    for i, metric in enumerate(base_metrics):
        plt.subplot(n_rows, n_cols, i + 1)
        
        # Plot Training Metric
        plt.plot(epochs_range, history[metric], label=f'Training {metric.title()}')
        
        # Plot Validation Metric (if exists)
        val_key = f"val_{metric}"
        if val_key in all_keys:
            plt.plot(epochs_range, history[val_key], label=f'Validation {metric.title()}', linestyle='--')
            
        plt.legend(loc='best')
        plt.title(f'{metric.title()}')
        plt.xlabel('Epochs')
        plt.grid(True, alpha=0.3)
        
        # Intelligent ticks
        if len(epochs_range) <= 20:
            plt.xticks(epochs_range[::2])
    
    plt.tight_layout()

    # 4. Save Main Figure to Run Directory
    if save_dir:
        # Ensure save_dir is a Path object
        save_dir = Path(save_dir)
        save_path = save_dir / "training_metrics.png"
        
        # Explicitly save the main_fig to avoid saving the LR plot instead if it exists
        main_fig.savefig(save_path)
        print(f"📊 Plots saved to: {save_path}")

    # Plot Learning Rate separately if it exists
    if 'lr' in all_keys:
        plt.figure(figsize=(10, 3))
        plt.plot(epochs_range, history['lr'], color='orange', label='Learning Rate')
        plt.title("Learning Rate Schedule")
        plt.xlabel("Epochs")
        plt.ylabel("LR")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
    plt.show()

# Example usage:
log_path = Path("<path_to_log.csv_file")
save_dir =  Path("<path_to_save_location")
plot_history_from_csv(log_path, save_dir=save_dir)

## Predicting Test Images

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# 1. Load the Best Model
best_model_path = checkpoint_dir / "best_model.keras"
print(f"Loading best model from: {best_model_path}")
final_model = tf.keras.models.load_model(best_model_path)

# 2. Quantitative Evaluation (Full Test Set)
print("\n---Quantitative Results (Test Set) ---")
test_results = final_model.evaluate(test_ds, verbose=1)
metrics_map = dict(zip(final_model.metrics_names, test_results))

print(f"\nFinal Test Results:")
# Dynamic key search to handle 'accuracy' vs 'acc' vs 'binary_accuracy'
loss_key = next((k for k in metrics_map.keys() if 'loss' in k), 'loss')
acc_key  = next((k for k in metrics_map.keys() if 'acc' in k), None)
prec_key = next((k for k in metrics_map.keys() if 'precision' in k), None)
rec_key  = next((k for k in metrics_map.keys() if 'recall' in k), None)

print(f"   • Loss:      {metrics_map[loss_key]:.4f}")
if acc_key: print(f"   • Accuracy:  {metrics_map[acc_key]:.2%}")
if prec_key: print(f"   • Precision: {metrics_map[prec_key]:.2%}")
if rec_key:  print(f"   • Recall:    {metrics_map[rec_key]:.2%}")


# --- 3. Qualitative Evaluation (Random Shuffle) ---
print("\n--- Visualizing Predictions (Random Batch) ---")

# Shuffle heavily (buffer=5000) to mix Positives and Negatives thoroughly
vis_ds = test_ds.unbatch().shuffle(buffer_size=5000).batch(32).take(1)
image_batch, label_batch = next(iter(vis_ds))

# Get predictions
predictions = final_model.predict(image_batch)
predicted_ids = (predictions > 0.5).astype(int).flatten()
true_ids = label_batch.numpy().astype(int).flatten()

# Setup Plot
plt.figure(figsize=(16, 10))
plt.suptitle(f"Random Test Batch Predictions", fontsize=16)

num_images = min(32, len(predicted_ids))
for i in range(num_images):
    ax = plt.subplot(4, 8, i + 1)
    
    # Display Image (Handle Grayscale vs RGB)
    img = image_batch[i].numpy().astype("uint8")
    if img.shape[-1] == 1:
        plt.imshow(img.squeeze(), cmap="gray")
    else:
        plt.imshow(img)
        
    # Determine Status
    is_correct = (predicted_ids[i] == true_ids[i])
    color = "green" if is_correct else "red"
    
    # Labels
    pred_lbl = "POS" if predicted_ids[i] == 1 else "NEG"
    true_lbl = "POS" if true_ids[i] == 1 else "NEG"
    
    # Probability: If Pred is POS, use p. If NEG, use 1-p.
    # This makes the number represent "Confidence in the Prediction"
    raw_prob = predictions[i][0]
    confidence = raw_prob if predicted_ids[i] == 1 else (1 - raw_prob)
    
    # Title: "Pred (True) \n Confidence"
    plt.title(f"{pred_lbl} ({true_lbl})\n{confidence:.1%}", color=color, fontsize=10, fontweight='bold')
    plt.axis("off")

plt.tight_layout()
vis_path = run_dir / "test_predictions_random.png"
plt.savefig(vis_path)
print(f"Visualization saved to: {vis_path}")

plt.show()